In [1]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *
from eval_functions import *

In [3]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)
save = True
eval_path = f"evaluate/batch1"
log_path = "evaluate/batch1/model_stats"
log_filename =  "evaluate/batch1/model_stats/missing_words_logfile.txt"
os.makedirs(eval_path, exist_ok=True)
os.makedirs(log_path, exist_ok=True)

model_output\Llama-3-70B-Instruct_5_shot
model_output\Llama-3-70B-Instruct_prompt_5_shot
model_output\Llama-3-8B-Instruct_4_shot
model_output\Llama-3-8B-Instruct_temp_4_shot_temp_1


### Model Output to nice JSON and Failure 

In [6]:
def process_files(model, save=save):
    input_dir = f"model_output/{model}/output/"
    output_dir = f"model_output/{model}/formatted/"
    failure_dir = f"model_output/{model}/failed/"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    files = os.listdir(input_dir)
    len_files = len(files)
    for filename in files:
        if filename.endswith('.json'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            failure_file_path = os.path.join(failure_dir, filename)
            try:
                file = read_json(input_file_path)
                #print(f"Processing file: {filename}")
                if save:
                    shutil.copy(input_file_path, output_file_path)
                    #save_json_to_file(file, output_file_path)
            except Exception as e:
                #print(f"Error processing file {filename}: {e}")
                if save:
                    shutil.copy(input_file_path, failure_file_path)
    return len_files

In [7]:
for model in models:
    print(model)
    # 1 - Preprocess Files
    num_out_files = process_files(model, save=save)
    # 2 - Evaluate Files
    p2_label_path = "../../chia_label/p2"
    ready_path = f"model_output/{model}/ready"
    failed_model_path = f"model_output/{model}/failed_inner"
    p2_model_formatted_path = f"model_output/{model}/formatted"
    for path in [ready_path, failed_model_path, p2_model_formatted_path]:
        os.makedirs(path, exist_ok=True)
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
    model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}
    
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    model_stats_path = os.path.join(log_path, f'{model}_stats.txt')
    missing_words_logfile = os.path.join(log_path, f'{model}_missing_words_logfile.txt')
    
    for nct in common_ncts:
        try:
            label_data = read_json(label_files[nct])
            model_data = read_json(model_files[nct])
            
            label_text = json_to_text(label_data)
            model_text = json_to_text(model_data)
        
            bleu_score = calculate_bleu(reference=label_text, hypothesis=model_text)
            jaccard_score = jaccard_similarity(label_text, model_text)
  
            print(f"BLEU Score: {bleu_score}")
            print(f"Jaccard Similarity: {jaccard_score}")
        
            label_structure = extract_logical_structure(label_data)
            model_structure = extract_logical_structure(model_data)
            # Versuche Wörter zu zählen
            label_raw_texts = extract_raw_texts(label_data)
            model_raw_texts = extract_raw_texts(model_data)
            label_words = extract_words(label_raw_texts)
            model_words = extract_words(model_raw_texts)
            missing_words = label_words - model_words
            missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
    
            with open(missing_words_logfile, 'a', encoding='utf-8') as outfile:
                outfile.write(f"{nct};   Missing Words: {round(missing_words_pct, 2)} %   ;  Is Subset:  {label_words.issubset(model_words)}\n")
                outfile.write("Label Text \n")
                outfile.write(f"{label_words} \n")
                outfile.write("\nMissing Words \n")
                outfile.write(f"{missing_words}\n")
                outfile.write("\nModel Text \n")
                outfile.write(f"{model_words}\n")
                outfile.write("\n\n")
            with open(model_stats_path, 'a', encoding='utf-8') as outfile:
                if not label_words.issubset(model_words):
                    outfile.write(f"{nct} [{model}] Missing Words: {round(missing_words_pct, 2)} %   ({len(missing_words)}) \n ")
    
            
            success_data.append({
                'NCT': nct,
                'label_AND': label_structure.get('AND', 0),
                'label_OR': label_structure.get('OR', 0),
                'label_NOT': label_structure.get('NOT', 0),
                'label_DEPTH': label_structure.get('depth', 0),
                'model_AND': model_structure.get('AND', 0),
                'model_OR': model_structure.get('OR', 0),
                'model_NOT': model_structure.get('NOT', 0),
                'model_DEPTH': model_structure.get('depth', 0),
                'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
                'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
                'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
                'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0,
                'num_out_files': num_out_files,
                'bleu_score': bleu_score,
                'jaccard_score': jaccard_score
            })
            labels.append(label_structure)
            predictions.append(model_structure)
            save and shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
        except Exception as e:
            #print(f"Error processing NCT {nct}: {e}")
            save and shutil.move(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))
    
    df_success = pd.DataFrame(success_data).set_index('NCT')
    df_success.to_csv(eval_path+f'/{model}_eval.csv')
    true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
    predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values
    
    metrics = {}
    for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
        y_true = true_values[:, i]
        y_pred = predicted_values[:, i]
    
        diffs = y_pred - y_true
        pct_greater = (diffs > 0).sum() / len(diffs) * 100
        pct_less = (diffs < 0).sum() / len(diffs) * 100
        pct_equal = (diffs == 0).sum() / len(diffs) * 100
    
        metrics[metric] = {
            'accuracy': round(accuracy_score(y_true, y_pred), 3),
            'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'pct_greater': round(pct_greater, 2),
            'pct_less': round(pct_less, 2),
            'pct_equal': round(pct_equal, 2)
            # 'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        metrics_df = pd.DataFrame(metrics).T
        num_nct_files = len(df_success)
        metrics_df['num_nct_files'] = num_nct_files
        metrics_df['model_name'] = model
        metrics_df['num_files'] = num_out_files
        metrics_df['mean_jacard'] = df_success['jaccard_score'].mean()
        metrics_df['mean_bleu'] = df_success['bleu_score'].mean()
        # Save Metrics to CSV
        metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))    

Llama-3-70B-Instruct_5_shot
BLEU Score: 0.6038238770367996
Jaccard Similarity: 0.9365079365079365
BLEU Score: 0.2755503900658499
Jaccard Similarity: 0.88
BLEU Score: 0.6959116104530774
Jaccard Similarity: 1.0
BLEU Score: 0.7726951981286995
Jaccard Similarity: 0.9761904761904762
BLEU Score: 0.6778361699458413
Jaccard Similarity: 0.8536585365853658
BLEU Score: 0.7913001060808317
Jaccard Similarity: 0.9411764705882353
BLEU Score: 0.2138023108559212
Jaccard Similarity: 0.896551724137931
BLEU Score: 0.651244408166087
Jaccard Similarity: 0.9285714285714286
BLEU Score: 0.5862863679861459
Jaccard Similarity: 0.9473684210526315
BLEU Score: 0.6728960751840685
Jaccard Similarity: 0.9672131147540983
BLEU Score: 0.6419557392152224
Jaccard Similarity: 1.0
BLEU Score: 0.5036470661983916
Jaccard Similarity: 0.9375
BLEU Score: 0.387122082640344
Jaccard Similarity: 0.9571428571428572
BLEU Score: 0.6884346301157118
Jaccard Similarity: 0.8518518518518519
BLEU Score: 0.9555630362682843
Jaccard Similarity: 

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 0.7013174720055447
Jaccard Similarity: 0.9622641509433962
BLEU Score: 0.7300161815198049
Jaccard Similarity: 0.9791666666666666
BLEU Score: 0.7421947953292999
Jaccard Similarity: 1.0
BLEU Score: 0.6144657016556727
Jaccard Similarity: 0.95
BLEU Score: 0.7136971717305517
Jaccard Similarity: 0.9215686274509803
BLEU Score: 0.7239888356504658
Jaccard Similarity: 0.9682539682539683
BLEU Score: 0.7736027043151418
Jaccard Similarity: 1.0
BLEU Score: 0.7298955711643803
Jaccard Similarity: 1.0
BLEU Score: 0.5555588037349539
Jaccard Similarity: 0.9655172413793104
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 0.7853457181276098
Jaccard Similarity: 1.0
BLEU Score: 0.7430154524353233
Jaccard Similarity: 1.0
BLEU Score: 0.6588987082934227
Jaccard Similarity: 0.9423076923076923
BLEU Score: 0.6620036925052327
Jaccard Similarity: 0.8125
BLEU Score: 0.6825874722138872
Jaccard Similarity: 1.0
BLEU Score: 0.7046242711977609
Jaccard Similarity: 1.0
BLEU Score: 0.8531362437541973
Jaccard Si

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 0.6897062909215053
Jaccard Similarity: 1.0
BLEU Score: 0.6964705665515707
Jaccard Similarity: 0.8148148148148148
BLEU Score: 0.8517240446030127
Jaccard Similarity: 1.0
BLEU Score: 0.45514585555887266
Jaccard Similarity: 0.8620689655172413
BLEU Score: 0.7681183176837931
Jaccard Similarity: 1.0
BLEU Score: 0.813641654071892
Jaccard Similarity: 1.0
BLEU Score: 0.7631223262784245
Jaccard Similarity: 0.8918918918918919
BLEU Score: 0.45438047183260133
Jaccard Similarity: 0.8405797101449275
BLEU Score: 0.4182639082534078
Jaccard Similarity: 0.86
BLEU Score: 0.6086645877120291
Jaccard Similarity: 0.8571428571428571
BLEU Score: 0.6590997066838998
Jaccard Similarity: 1.0
BLEU Score: 0.8445634383579881
Jaccard Similarity: 0.95
BLEU Score: 0.9082662684476391
Jaccard Similarity: 1.0
BLEU Score: 0.6032521213935551
Jaccard Similarity: 0.9534883720930233
BLEU Score: 0
Jaccard Similarity: 0.0
BLEU Score: 0.846393869815524
Jaccard Similarity: 1.0
BLEU Score: 0.7044782660388055
Jaccard Simila

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 4.1324064904861155e-05
Jaccard Similarity: 0.47619047619047616
BLEU Score: 0.06193250743057741
Jaccard Similarity: 0.6363636363636364
BLEU Score: 0.1441685708809707
Jaccard Similarity: 0.8709677419354839
BLEU Score: 0.48288876755053267
Jaccard Similarity: 0.9130434782608695
BLEU Score: 0.09259549653382697
Jaccard Similarity: 0.6428571428571429
BLEU Score: 5.14828679008516e-155
Jaccard Similarity: 0.6176470588235294
BLEU Score: 0.047147946854309064
Jaccard Similarity: 0.82
BLEU Score: 0.20434123055097414
Jaccard Similarity: 0.7241379310344828
BLEU Score: 0.6919082322305816
Jaccard Similarity: 0.9375
BLEU Score: 0.5502669213361729
Jaccard Similarity: 0.9310344827586207
BLEU Score: 0.4318912122719855
Jaccard Similarity: 0.8857142857142857
BLEU Score: 0.684694991173804
Jaccard Similarity: 1.0
BLEU Score: 0.5851302012327348
Jaccard Similarity: 0.9607843137254902
BLEU Score: 0.0008692361372977963
Jaccard Similarity: 0.6724137931034483
BLEU Score: 7.070696784820904e-78
Jaccard Sim

C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\e-aut\anaconda3\lib\site-packages\nltk\translate\bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnin

BLEU Score: 0.21133320213612583
Jaccard Similarity: 0.696969696969697
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 0.33409085098947405
Jaccard Similarity: 0.9310344827586207
BLEU Score: 0
Jaccard Similarity: 0.0
BLEU Score: 0.4403735172581674
Jaccard Similarity: 0.9024390243902439
BLEU Score: 0.5503518419947444
Jaccard Similarity: 0.9795918367346939
BLEU Score: 1.0
Jaccard Similarity: 1.0
BLEU Score: 0.5749296974573944
Jaccard Similarity: 0.918918918918919
BLEU Score: 0.5136434269817112
Jaccard Similarity: 1.0
BLEU Score: 0.3058220915064213
Jaccard Similarity: 0.8125
BLEU Score: 0.02831996552400536
Jaccard Similarity: 0.6818181818181818
BLEU Score: 0.0015206125179657603
Jaccard Similarity: 0.5932203389830508
BLEU Score: 0.23139349439155177
Jaccard Similarity: 0.8571428571428571
BLEU Score: 0.07481603425080126
Jaccard Similarity: 0.9152542372881356
BLEU Score: 0.7208751523704692
Jaccard Similarity: 1.0
BLEU Score: 0.24623953025272619
Jaccard Similarity: 0.8620689655172413
BLEU Sc

In [8]:
metrics_df# NCT02935855_inc

,accuracy,precision,recall,f1_score,pct_greater,pct_less,pct_equal,num_nct_files,model_name,num_files,mean_jacard,mean_bleu
AND,0.203,0.149,0.203,0.165,25.42,54.24,20.34,118,Llama-3-8B-Instruct_temp_4_shot_temp_1,304,0.848453,0.367716
OR,0.331,0.389,0.331,0.347,38.98,27.97,33.05,118,Llama-3-8B-Instruct_temp_4_shot_temp_1,304,0.848453,0.367716
NOT,0.746,0.674,0.746,0.694,5.08,20.34,74.58,118,Llama-3-8B-Instruct_temp_4_shot_temp_1,304,0.848453,0.367716
DEPTH,0.203,0.261,0.203,0.202,25.42,54.24,20.34,118,Llama-3-8B-Instruct_temp_4_shot_temp_1,304,0.848453,0.367716
